In [60]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

In [61]:
df = pd.read_csv(r"fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [62]:
x = df.iloc[:, 1:]
y = df.iloc[:,0]

In [63]:
x

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,0,0,0,0,0,0,0,0,1,...,69,12,0,0,0,0,0,0,0,0
5996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,0,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [64]:
y

0       9
1       7
2       0
3       8
4       8
       ..
5995    1
5996    5
5997    8
5998    4
5999    8
Name: label, Length: 6000, dtype: int64

In [65]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [66]:
x_train = x_train/255.0
x_test = x_test/255.0

In [67]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [68]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [69]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [70]:
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 10),
            nn.Softmax()
        )

    def forward(self, features):
        return self.network(features)    


In [71]:
learning_rate = 0.1
epoch = 100
loss_function = nn.CrossEntropyLoss()


In [72]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [73]:
model = MyNN(x_train.shape[1])

model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [74]:
for epoch in range(epoch):
    total_loss = 0
    for features, lables in train_loader:
        features, lables = features.to(device), lables.to(device)
        y_pred = model(features)

        loss = loss_function(y_pred, lables)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)    

    print(epoch+1, avg_loss)    

c:\Desktop\PyTorch\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


1 2.2939366165796917
2 2.215456580320994
3 2.0038154300053916
4 1.8799638183911642
5 1.8423205176989237
6 1.8150359161694845
7 1.7867086768150329
8 1.7704091954231262
9 1.7605207173029582
10 1.7538140042622885
11 1.7461684679985046
12 1.742048330307007
13 1.7383143321673076
14 1.7358382582664489
15 1.7339140892028808
16 1.7301101636886598
17 1.727510118484497
18 1.7269529048601786
19 1.7237367351849875
20 1.7230437676111856
21 1.720708932876587
22 1.7193103075027465
23 1.7177258626619976
24 1.7166907986005147
25 1.7164677898089091
26 1.7142783331871032
27 1.714674190680186
28 1.7127852527300518
29 1.711663802464803
30 1.7115174023310344
31 1.7107474557558695
32 1.7109228340784708
33 1.7079318610827128
34 1.708340009053548
35 1.7067360917727152
36 1.7065723093350729
37 1.705029456615448
38 1.7062893350919088
39 1.7039525445302328
40 1.7041495243708293
41 1.7030398400624593
42 1.7023995423316955
43 1.7010893964767455
44 1.7012737711270651
45 1.7013387235005697
46 1.7012229919433595
47 1.

In [78]:
model.eval()

MyNN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
    (5): Softmax(dim=None)
  )
)

In [79]:
total = 0
correct = 0


with torch.no_grad():
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=lables.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.7720833333333333


In [80]:
total = 0
correct = 0


with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=lables.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.7105263157894737
